# Task 1 — Pairwise Distances
Set up the notebook environment for computing Hamming and p-distance matrices on personal Lab 1 sequences.

## 1. Initialize Project Environment
Import dependencies, confirm versions, and run a quick sanity check to ensure Biopython can parse Lab 1 FASTA files.

In [12]:
from __future__ import annotations

import itertools
import logging
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
print("numpy", np.__version__)
try:
    import Bio

    print("biopython", Bio.__version__)
except Exception as exc:  # pragma: no cover
    logging.error("Biopython import failed: %s", exc)


pandas 2.2.3
numpy 2.1.3
biopython 1.85


In [13]:
DATA_ROOT = Path("../../../data/work/AndreiCod/lab01")
FASTAS = list(DATA_ROOT.glob("*.fa*"))
print("Discovered FASTA files:")
for path in FASTAS:
    print("-", path, "size", path.stat().st_size / 1e6, "MB")

assert FASTAS, "No Lab 1 FASTA files found; double-check the path."


Discovered FASTA files:
- ../../../data/work/AndreiCod/lab01/my_tp53.fa size 169.961179 MB
- ../../../data/work/AndreiCod/lab01/nm000546.fa size 0.002628 MB


## 2. Define Configuration Parameters
Centralize file paths, sequence selection, and distance options so reruns stay deterministic and reproducible.

In [14]:
from dataclasses import dataclass, asdict


@dataclass
class DistanceConfig:
    fasta_path: Path
    selected_ids: List[str]
    max_seqs: int = 5
    truncate_to_min: bool = True

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["fasta_path"] = str(info["fasta_path"])
        return info


CONFIG = DistanceConfig(
    fasta_path=DATA_ROOT / "my_tp53.fa",
    selected_ids=[],  # leave empty to auto-select first `max_seqs`
    max_seqs=4,
    truncate_to_min=True,
)

CONFIG.describe()

{'fasta_path': '../../../data/work/AndreiCod/lab01/my_tp53.fa',
 'selected_ids': [],
 'max_seqs': 4,
 'truncate_to_min': True}

In [15]:
def select_records(cfg: DistanceConfig) -> List[SeqIO.SeqRecord]:
    """Load FASTA records and select based on IDs or order."""
    if not cfg.fasta_path.exists():  # pragma: no cover
        raise FileNotFoundError(cfg.fasta_path)

    recs = list(SeqIO.parse(cfg.fasta_path, "fasta"))
    if not recs:
        raise ValueError("No sequences found in FASTA")

    if cfg.selected_ids:
        filtered = [r for r in recs if r.id in cfg.selected_ids]
    else:
        filtered = recs[: cfg.max_seqs]

    if len(filtered) < 3:
        raise ValueError("Need at least 3 sequences for the distance matrix")

    logging.info("Selected %d sequences", len(filtered))
    return filtered


selected_records = select_records(CONFIG)
[id_ for id_ in [rec.id for rec in selected_records]]

[INFO] Selected 3 sequences


['NG_017013.2', 'NC_060941.1', 'NC_000017.11']

## 3. Implement Core Functionality
Build utilities for computing Hamming and p-distance scores, plus routines to format them as an upper-triangular matrix.

In [16]:
def truncate_pair(a: str, b: str, enabled: bool = True) -> Tuple[str, str, int]:
    if not enabled:
        return a, b, len(a)
    L = min(len(a), len(b))
    return a[:L], b[:L], L


def hamming_distance(a: str, b: str) -> int:
    if len(a) != len(b):  # pragma: no cover
        raise ValueError("Hamming distance requires equal-length strings")
    return sum(ch1 != ch2 for ch1, ch2 in zip(a, b))


def compute_pair_metrics(
    records: List[SeqIO.SeqRecord], truncate: bool = True
) -> pd.DataFrame:
    matrix_rows = []
    for rec_i, rec_j in itertools.combinations(records, 2):
        seq_i, seq_j, used_len = truncate_pair(
            str(rec_i.seq), str(rec_j.seq), enabled=truncate
        )
        d_hamming = hamming_distance(seq_i, seq_j)
        d_p = d_hamming / used_len if used_len else np.nan
        matrix_rows.append(
            {
                "seq_a": rec_i.id,
                "seq_b": rec_j.id,
                "used_len": used_len,
                "hamming": d_hamming,
                "p_distance": d_p,
            }
        )
    return pd.DataFrame(matrix_rows)


pairwise_df = compute_pair_metrics(selected_records, truncate=CONFIG.truncate_to_min)
pairwise_df

,seq_a,seq_b,used_len,hamming,p_distance
0,NG_017013.2,NC_060941.1,32772,24498,0.747528
1,NG_017013.2,NC_000017.11,32772,32772,1.000000
2,NC_060941.1,NC_000017.11,83257441,62200462,0.747086


In [17]:
def upper_triangle(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    pivot = df.pivot(index="seq_a", columns="seq_b", values=value_col)
    return pivot


hamming_matrix = upper_triangle(pairwise_df, "hamming")
p_distance_matrix = upper_triangle(pairwise_df, "p_distance")

hamming_matrix

seq_b,NC_000017.11,NC_060941.1
seq_a,,
NC_060941.1,62200462.0,NaN
NG_017013.2,32772.0,24498.0


In [18]:
p_distance_matrix

seq_b,NC_000017.11,NC_060941.1
seq_a,,
NC_060941.1,0.747086,NaN
NG_017013.2,1.000000,0.747528


## 4. Validate with Unit Tests
Quick inline assertions guard the distance helpers so regressions are easy to spot before exporting results.

In [19]:
def test_hamming_distance():
    assert hamming_distance("AAAA", "AAAT") == 1
    assert hamming_distance("AC", "GT") == 2


def test_truncate_pair():
    a, b, used = truncate_pair("AAAA", "AA", True)
    assert used == 2 and a == "AA" and b == "AA"


def test_compute_pair_metrics():
    dummy_records = [
        SeqIO.SeqRecord(seq=Seq("AAAA"), id="A"),
        SeqIO.SeqRecord(seq=Seq("AAAT"), id="B"),
        SeqIO.SeqRecord(seq=Seq("AATT"), id="C"),
    ]
    df = compute_pair_metrics(dummy_records, truncate=True)
    assert set(df.columns) == {"seq_a", "seq_b", "used_len", "hamming", "p_distance"}
    assert len(df) == 3


test_hamming_distance()
test_truncate_pair()
test_compute_pair_metrics()
print("All inline tests passed.")

All inline tests passed.


## 5. Analyze Performance Metrics
Track runtime and memory characteristics of the distance calculations to justify scalability.

In [20]:
import time


def benchmark(fn, *args, repeat: int = 3, **kwargs) -> Dict[str, float]:
    durations = []
    for _ in range(repeat):
        start = time.perf_counter()
        fn(*args, **kwargs)
        durations.append(time.perf_counter() - start)
    arr = np.array(durations)
    return {
        "mean_s": arr.mean(),
        "std_s": arr.std(),
        "runs": repeat,
    }


metrics = benchmark(compute_pair_metrics, selected_records, CONFIG.truncate_to_min)
metrics

{'mean_s': np.float64(3.3872390326666086),
 'std_s': np.float64(0.10049571097611171),
 'runs': 3}

In [21]:
pd.DataFrame([metrics])

,mean_s,std_s,runs
0,3.387239,0.100496,3


In [22]:
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pairwise_df.to_csv(OUTPUT_DIR / "task1_pairwise_distances.csv", index=False)
p_distance_matrix.to_csv(OUTPUT_DIR / "task1_p_distance_matrix.csv")
print("Artifacts saved to", OUTPUT_DIR)

Artifacts saved to artifacts
